# STV baseline — Docker Jupyter lab
컨테이너에 설치된 런타임(torch / unsloth / trl)을 그대로 사용합니다. uv venv는 쓰지 않습니다.

코드는 호스트에서 `bash scripts/lab_sync.sh`로 `/workspace/scene-text-vision-repo`에 복사됩니다. 코드를 고치면 다시 sync하세요.
학습/추론은 단계마다 별도 프로세스로 실행해서 끝나면 VRAM이 해제됩니다.

In [ ]:
%cd /workspace/scene-text-vision-repo
!bash scripts/setup.sh

In [ ]:
CONFIG = "configs/lab.toml"
OUT = "outputs/qwen2_5_vl_3b_lora"
EXTRA = "--max-train-samples 200"   # 전체 학습: --max-train-samples 0

In [ ]:
!HF_HUB_OFFLINE=1 python -m stv.train --config $CONFIG --output-dir $OUT $EXTRA

In [ ]:
!HF_HUB_OFFLINE=1 python -m stv.inference --config $CONFIG --output-dir $OUT --split dev

In [ ]:
!HF_HUB_OFFLINE=1 python -m stv.inference --config $CONFIG --output-dir $OUT --split test

## 결과 확인

In [ ]:
import numpy as np, pandas as pd
from stv.config import load_config
from stv.data import CHOICES, load_split

cfg = load_config(CONFIG, output_dir=OUT)
dev = load_split(cfg, "dev")
pred = pd.read_csv(f"{OUT}/submission_dev.csv")
dev["pred"] = pred["answer"].values
dev["conf"] = np.load(f"{OUT}/dev_probs.npz")["avg"].max(1)
print("dev acc:", (dev["pred"] == dev["answer"]).mean())
dev[dev["pred"] != dev["answer"]].sort_values("conf", ascending=False).head(10)